# Isotope Peak Detection in NIST20 and NIST23

Validates `detect_isotope_peaks` from `fragnnet.utils.spec_utils` on both reference datasets.

**Questions answered:**
- What fraction of spectra contain isotope peaks?
- What fraction of all peaks are isotope peaks?
- How does this break down by instrument type, fragmentation mode, and precursor type?
- Do NIST20 and NIST23 differ in isotope prevalence?

In [ ]:
import gc
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from fragnnet.utils.spec_utils import detect_isotope_peaks

PROC_DP = Path("../data/proc")
SAMPLE_N = 100_000   # spectra per dataset; set to None to use all
RANDOM_SEED = 42

SPEC_COLS = ["spec_id", "mol_id", "inst_type", "frag_mode", "prec_type",
             "ion_mode", "ace", "nce", "prec_mz", "peaks"]

## Helper — per-spectrum isotope stats

In [ ]:
import math
import re
from functools import lru_cache

NEUTRON_MASS = 1.008665  # same constant as spec_utils

_M1_PER_ATOM: dict[str, float] = {
    "C": 0.01103, "H": 0.000149, "N": 0.00368,
    "O": 0.000404, "S": 0.00421, "Si": 0.04683,
}


@lru_cache(maxsize=8192)
def _parse_formula(formula: str) -> tuple[tuple[str, int], ...]:
    return tuple(
        (elem, int(count) if count else 1)
        for elem, count in re.findall(r"([A-Z][a-z]?)(\d*)", formula)
    )


def expected_m1_ratio(formula: str) -> float:
    counts = dict(_parse_formula(formula))
    return sum(counts.get(elem, 0) * contrib for elem, contrib in _M1_PER_ATOM.items())


def detect_isotope_peaks_formula_aware(
    mzs: np.ndarray,
    ints: np.ndarray,
    prec_mz: float,
    formula: str,
    mz_tol: float = 0.02,
    max_isotope: int = 2,
    lo_factor: float = 0.3,
    hi_factor: float = 3.0,
) -> np.ndarray:
    mzs = np.asarray(mzs, dtype=np.float64)
    ints = np.asarray(ints, dtype=np.float64)
    n = len(mzs)
    is_isotope = np.zeros(n, dtype=bool)
    if n == 0 or prec_mz <= 0:
        return is_isotope
    mol_m1 = expected_m1_ratio(formula)
    sort_idx = np.argsort(mzs)
    sorted_mzs = mzs[sort_idx]
    sorted_ints = ints[sort_idx]
    for i in range(n):
        frag_mz = sorted_mzs[i]
        frag_int = sorted_ints[i]
        scale = min(frag_mz / prec_mz, 1.0)
        for k in range(1, max_isotope + 1):
            target_mz = frag_mz - k * NEUTRON_MASS
            lo = np.searchsorted(sorted_mzs, target_mz - mz_tol, side="left")
            hi_idx = np.searchsorted(sorted_mzs, target_mz + mz_tol, side="right")
            if lo >= hi_idx:
                continue
            best_m0_int = sorted_ints[lo:hi_idx].max()
            if best_m0_int <= 0:
                continue
            obs_ratio = frag_int / best_m0_int
            exp_ratio = max((mol_m1 * scale) ** k / math.factorial(k), 1e-4)
            if lo_factor * exp_ratio <= obs_ratio <= hi_factor * exp_ratio:
                is_isotope[sort_idx[i]] = True
                break
    return is_isotope


def compute_isotope_stats(
    peaks, mz_tol: float = 0.02, max_isotope: int = 3, check_intensity: bool = True
) -> dict:
    if not peaks:
        return {"n_peaks": 0, "n_isotope": 0, "frac_isotope": np.nan, "has_isotope": False}
    mzs = np.array([p[0] for p in peaks], dtype=np.float64)
    ints = np.array([p[1] for p in peaks], dtype=np.float64)
    mask = detect_isotope_peaks(mzs, ints, mz_tol=mz_tol, max_isotope=max_isotope, check_intensity=check_intensity)
    n_iso = int(mask.sum())
    n_total = len(mzs)
    return {"n_peaks": n_total, "n_isotope": n_iso,
            "frac_isotope": n_iso / n_total if n_total > 0 else np.nan, "has_isotope": n_iso > 0}


def compute_isotope_stats_formula_aware(
    peaks, prec_mz: float, formula: str,
    mz_tol: float = 0.02, max_isotope: int = 2,
    lo_factor: float = 0.3, hi_factor: float = 3.0,
) -> dict:
    if not peaks or not isinstance(formula, str) or prec_mz <= 0:
        return {"n_peaks": 0, "n_isotope_fa": 0, "frac_isotope_fa": np.nan, "has_isotope_fa": False}
    mzs = np.array([p[0] for p in peaks], dtype=np.float64)
    ints = np.array([p[1] for p in peaks], dtype=np.float64)
    mask = detect_isotope_peaks_formula_aware(
        mzs, ints, prec_mz=prec_mz, formula=formula,
        mz_tol=mz_tol, max_isotope=max_isotope, lo_factor=lo_factor, hi_factor=hi_factor,
    )
    n_iso = int(mask.sum())
    n_total = len(mzs)
    return {"n_peaks": n_total, "n_isotope_fa": n_iso,
            "frac_isotope_fa": n_iso / n_total if n_total > 0 else np.nan, "has_isotope_fa": n_iso > 0}


def run_isotope_analysis(spec_df: pd.DataFrame) -> pd.DataFrame:
    """Compute isotope stats on an already-sampled spec_df. No internal sampling."""
    print(f"  Computing isotope stats on {len(spec_df):,} spectra…")
    stats = spec_df["peaks"].apply(compute_isotope_stats).apply(pd.Series)
    meta_cols = [c for c in ["spec_id", "mol_id", "inst_type", "frag_mode", "prec_type",
                              "ion_mode", "ace", "nce", "prec_mz", "formula"] if c in spec_df.columns]
    result = pd.concat([spec_df[meta_cols].reset_index(drop=True), stats], axis=1)
    print(f"  Done. {result['has_isotope'].sum():,} / {len(result):,} spectra have ≥1 isotope peak "
          f"({100 * result['has_isotope'].mean():.1f}%)")
    return result

## NIST20

In [ ]:
print("Loading NIST20…")
_mol20 = pd.read_pickle(PROC_DP / "nist20/mol_df.pkl")[["mol_id", "formula", "exact_mw"]]
_spec20 = pd.read_pickle(PROC_DP / "nist20/spec_df.pkl")[SPEC_COLS].merge(_mol20, on="mol_id", how="left")
del _mol20
print(f"  Total spectra: {len(_spec20):,}  |  formula coverage: {_spec20['formula'].notna().mean():.1%}")
print(_spec20["inst_type"].value_counts().to_string())

# Sample immediately — frees ~90% of peak data
if SAMPLE_N is not None and len(_spec20) > SAMPLE_N:
    spec20 = _spec20.sample(SAMPLE_N, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"  Sampled {SAMPLE_N:,} spectra")
else:
    spec20 = _spec20
del _spec20; gc.collect()

stats20 = run_isotope_analysis(spec20)

## NIST23

In [ ]:
print("Loading NIST23…")
_mol23 = pd.read_pickle(PROC_DP / "nist23/mol_df.pkl")[["mol_id", "formula", "exact_mw"]]
_spec23 = pd.read_pickle(PROC_DP / "nist23/spec_df.pkl")[SPEC_COLS].merge(_mol23, on="mol_id", how="left")
del _mol23
print(f"  Total spectra: {len(_spec23):,}  |  formula coverage: {_spec23['formula'].notna().mean():.1%}")
print(_spec23["inst_type"].value_counts().to_string())

if SAMPLE_N is not None and len(_spec23) > SAMPLE_N:
    spec23 = _spec23.sample(SAMPLE_N, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"  Sampled {SAMPLE_N:,} spectra")
else:
    spec23 = _spec23
del _spec23; gc.collect()

stats23 = run_isotope_analysis(spec23)

## Summary statistics

In [ ]:
def summary_table(stats: pd.DataFrame, label: str) -> pd.DataFrame:
    rows = []
    rows.append({
        "dataset": label, "group": "ALL", "value": "all",
        "n_spectra": len(stats),
        "pct_has_isotope": 100 * stats["has_isotope"].mean(),
        "mean_frac_isotope": stats["frac_isotope"].mean(),
        "median_n_isotope": stats["n_isotope"].median(),
    })
    for group_col in ["inst_type", "frag_mode", "prec_type"]:
        if group_col not in stats.columns:
            continue
        for val, sub in stats.groupby(group_col, dropna=True):
            rows.append({
                "dataset": label, "group": group_col, "value": val,
                "n_spectra": len(sub),
                "pct_has_isotope": 100 * sub["has_isotope"].mean(),
                "mean_frac_isotope": sub["frac_isotope"].mean(),
                "median_n_isotope": sub["n_isotope"].median(),
            })
    return pd.DataFrame(rows)


tbl20 = summary_table(stats20, "NIST20")
tbl23 = summary_table(stats23, "NIST23")
tbl_all = pd.concat([tbl20, tbl23], ignore_index=True)

with pd.option_context("display.float_format", "{:.3f}".format, "display.max_rows", 60):
    display(tbl_all)

## Plots

In [ ]:
# ── Overview: % spectra with isotope peaks by inst_type ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

for ax, (stats, label) in zip(axes, [(stats20, "NIST20"), (stats23, "NIST23")]):
    grp = (
        stats.groupby("inst_type", dropna=True)["has_isotope"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )
    grp["pct"] = grp["mean"] * 100
    bars = ax.bar(range(len(grp)), grp["pct"], color="steelblue", alpha=0.8)
    for i, row in grp.iterrows():
        ax.text(i, row["pct"] + 0.3, f"n={row['count']:,}", ha="center", fontsize=7)
    ax.set_xticks(range(len(grp)))
    ax.set_xticklabels(grp["inst_type"], rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("% spectra with ≥1 isotope peak")
    ax.set_title(f"{label} — by instrument type")
    ax.set_ylim(0, max(grp["pct"].max() * 1.15, 5))

plt.suptitle("Fraction of spectra with detected isotope peaks", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── By frag_mode ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, (stats, label) in zip(axes, [(stats20, "NIST20"), (stats23, "NIST23")]):
    if "frag_mode" not in stats.columns:
        ax.set_title(f"{label} — frag_mode not available")
        continue
    grp = (
        stats.groupby("frag_mode", dropna=True)["has_isotope"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )
    grp["pct"] = grp["mean"] * 100
    ax.bar(range(len(grp)), grp["pct"], color="darkorange", alpha=0.8)
    for i, row in grp.iterrows():
        ax.text(i, row["pct"] + 0.3, f"n={row['count']:,}", ha="center", fontsize=7)
    ax.set_xticks(range(len(grp)))
    ax.set_xticklabels(grp["frag_mode"], rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("% spectra with ≥1 isotope peak")
    ax.set_title(f"{label} — by fragmentation mode")
    ax.set_ylim(0, max(grp["pct"].max() * 1.15, 5))

plt.suptitle("Isotope peak prevalence by fragmentation mode", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── By prec_type ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

for ax, (stats, label) in zip(axes, [(stats20, "NIST20"), (stats23, "NIST23")]):
    grp = (
        stats.groupby("prec_type", dropna=True)["has_isotope"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )
    grp["pct"] = grp["mean"] * 100
    ax.bar(range(len(grp)), grp["pct"], color="mediumseagreen", alpha=0.8)
    for i, row in grp.iterrows():
        ax.text(i, row["pct"] + 0.3, f"n={row['count']:,}", ha="center", fontsize=7)
    ax.set_xticks(range(len(grp)))
    ax.set_xticklabels(grp["prec_type"], rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("% spectra with ≥1 isotope peak")
    ax.set_title(f"{label} — by precursor type")
    ax.set_ylim(0, max(grp["pct"].max() * 1.15, 5))

plt.suptitle("Isotope peak prevalence by precursor type", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution: number of isotope peaks per spectrum ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (stats, label) in zip(axes, [(stats20, "NIST20"), (stats23, "NIST23")]):
    counts = stats["n_isotope"].value_counts().sort_index()
    # Show 0..10, group rest
    MAX_SHOW = 10
    low = counts[counts.index <= MAX_SHOW]
    high_sum = counts[counts.index > MAX_SHOW].sum()
    xs = list(low.index) + ([f">{MAX_SHOW}"] if high_sum > 0 else [])
    ys = list(low.values) + ([high_sum] if high_sum > 0 else [])
    ax.bar(range(len(xs)), ys, color="slateblue", alpha=0.8)
    ax.set_xticks(range(len(xs)))
    ax.set_xticklabels(xs, fontsize=9)
    ax.set_xlabel("Number of isotope peaks per spectrum")
    ax.set_ylabel("Count of spectra")
    ax.set_title(f"{label}")
    ax.set_yscale("log")

plt.suptitle("Distribution of isotope peak count per spectrum (log scale)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution: fraction of peaks that are isotope peaks ───────────────────
# Only for spectra that have ≥1 isotope peak
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (stats, label) in zip(axes, [(stats20, "NIST20"), (stats23, "NIST23")]):
    sub = stats[stats["has_isotope"] & stats["frac_isotope"].notna()]["frac_isotope"]
    ax.hist(sub, bins=40, range=(0, 1), color="tomato", alpha=0.7, edgecolor="white")
    ax.axvline(sub.median(), color="black", lw=1.5, ls="--", label=f"median={sub.median():.3f}")
    ax.axvline(sub.mean(), color="navy", lw=1.5, ls="-.", label=f"mean={sub.mean():.3f}")
    ax.set_xlabel("Fraction of peaks that are isotope peaks")
    ax.set_ylabel("Count of spectra")
    ax.set_title(f"{label} (n={len(sub):,} spectra with isotopes)")
    ax.legend(fontsize=9)

plt.suptitle("Isotope fraction distribution (spectra with ≥1 isotope peak)", y=1.02)
plt.tight_layout()
plt.show()

## Example spectra with isotope peaks highlighted

In [ ]:
def plot_spectrum_with_isotopes(
    ax, peaks, title: str = "", mz_tol: float = 0.02, max_isotope: int = 3
) -> None:
    """Plot a spectrum with isotope peaks highlighted in red."""
    mzs = np.array([p[0] for p in peaks], dtype=np.float64)
    ints = np.array([p[1] for p in peaks], dtype=np.float64)
    ints_norm = ints / ints.max() * 100
    mask = detect_isotope_peaks(mzs, ints, mz_tol=mz_tol, max_isotope=max_isotope)

    for mz, intensity, is_iso in zip(mzs, ints_norm, mask):
        color = "red" if is_iso else "steelblue"
        ax.vlines(mz, 0, intensity, color=color, lw=1.2, alpha=0.85)

    n_iso = mask.sum()
    ax.set_xlabel("m/z")
    ax.set_ylabel("Relative intensity (%)")
    ax.set_title(f"{title}  [{n_iso}/{len(mzs)} isotope peaks]", fontsize=9)
    # Legend proxies
    from matplotlib.lines import Line2D
    ax.legend(
        handles=[
            Line2D([0], [0], color="steelblue", lw=2, label="fragment"),
            Line2D([0], [0], color="red", lw=2, label="isotope"),
        ],
        fontsize=7,
    )


# Pull 6 spectra that have isotope peaks (from NIST23)
example_ids = stats23[stats23["n_isotope"] >= 2]["spec_id"].sample(6, random_state=0).values
ex_peaks = (
    spec23[spec23["spec_id"].isin(example_ids)][["spec_id", "peaks", "inst_type", "prec_type"]]
    .set_index("spec_id")
    .loc[example_ids]
)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (spec_id, row) in zip(axes.flat, ex_peaks.iterrows()):
    title = f"NIST23 spec {spec_id} | {row['inst_type']} | {row['prec_type']}"
    plot_spectrum_with_isotopes(ax, row["peaks"], title=title)
plt.suptitle("Example spectra: fragment peaks (blue) vs isotope peaks (red)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Same for NIST20
example_ids20 = stats20[stats20["n_isotope"] >= 2]["spec_id"].sample(6, random_state=1).values
ex_peaks20 = (
    spec20[spec20["spec_id"].isin(example_ids20)][["spec_id", "peaks", "inst_type", "prec_type"]]
    .set_index("spec_id")
    .loc[example_ids20]
)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (spec_id, row) in zip(axes.flat, ex_peaks20.iterrows()):
    title = f"NIST20 spec {spec_id} | {row['inst_type']} | {row['prec_type']}"
    plot_spectrum_with_isotopes(ax, row["peaks"], title=title)
plt.suptitle("NIST20 example spectra: fragment peaks (blue) vs isotope peaks (red)", y=1.01)
plt.tight_layout()
plt.show()

## NIST20 vs NIST23 comparison

In [ ]:
# Side-by-side comparison by inst_type (only types present in both datasets)
def pct_by_group(stats, col):
    return (
        stats.groupby(col, dropna=True)["has_isotope"]
        .mean()
        .mul(100)
        .rename("pct")
        .reset_index()
    )


g20 = pct_by_group(stats20, "inst_type").rename(columns={"pct": "NIST20"})
g23 = pct_by_group(stats23, "inst_type").rename(columns={"pct": "NIST23"})
comp = g20.merge(g23, on="inst_type", how="outer").set_index("inst_type").fillna(0)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(comp))
w = 0.35
ax.bar(x - w / 2, comp["NIST20"], w, label="NIST20", color="steelblue", alpha=0.85)
ax.bar(x + w / 2, comp["NIST23"], w, label="NIST23", color="darkorange", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(comp.index, rotation=20, ha="right")
ax.set_ylabel("% spectra with ≥1 isotope peak")
ax.set_title("Isotope prevalence: NIST20 vs NIST23 by instrument type")
ax.legend()
plt.tight_layout()
plt.show()

# Print numeric table
print(comp.round(2).to_string())

## Sensitivity to `mz_tol` and `check_intensity`

In [ ]:
# How sensitive are results to the tolerance and intensity-check flag?
# Use a small NIST23 sub-sample for speed.
SENS_N = 10_000
sens_sample = spec23.sample(SENS_N, random_state=0)

configs = [
    {"mz_tol": 0.01, "check_intensity": True},
    {"mz_tol": 0.02, "check_intensity": True},   # default
    {"mz_tol": 0.05, "check_intensity": True},
    {"mz_tol": 0.02, "check_intensity": False},
]

rows = []
for cfg in configs:
    pct = sens_sample["peaks"].apply(
        lambda p: compute_isotope_stats(p, **cfg)["has_isotope"]
    ).mean() * 100
    mean_n = sens_sample["peaks"].apply(
        lambda p: compute_isotope_stats(p, **cfg)["n_isotope"]
    ).mean()
    rows.append({**cfg, "pct_has_isotope": pct, "mean_n_isotope": mean_n})

sens_df = pd.DataFrame(rows)
print("Sensitivity analysis (NIST23 sample, n={:,}):".format(SENS_N))
display(sens_df.round(3))

## Formula-aware isotope detection

Because `formula` is available in `mol_df` for every spectrum, we can replace the naive
`M0_intensity ≥ M1_intensity` heuristic with a **quantitative intensity-ratio check**:

For a fragment peak at m/z `X` from precursor at m/z `P` with formula `F`:

```
scale      = X / P                  # fraction of molecule's mass in this fragment
exp_ratio  = M1(F) × scale          # expected M+1/M+0 ratio for this fragment
```

A peak is accepted as isotope only if `0.3 × exp_ratio ≤ obs_ratio ≤ 3.0 × exp_ratio`.

This **eliminates false positives** where two unrelated fragment peaks accidentally sit
~1 Da apart but have very different intensities (e.g. a 10% peak next to a 90% peak when
the formula predicts only a ~5% M+1).

In [ ]:
# Run both detectors on a shared NIST23 sample (10k for speed)
FA_N = 10_000
fa_sample = spec23.sample(FA_N, random_state=7).reset_index(drop=True)

# Naive detection (default settings)
naive_stats = fa_sample["peaks"].apply(compute_isotope_stats).apply(pd.Series)

# Formula-aware detection — requires formula and prec_mz
has_formula = fa_sample["formula"].notna() & (fa_sample["prec_mz"].fillna(0) > 0)
print(f"Spectra with formula + prec_mz: {has_formula.sum():,} / {FA_N:,} "
      f"({100 * has_formula.mean():.1f}%)")

fa_stats = fa_sample.apply(
    lambda row: compute_isotope_stats_formula_aware(
        row["peaks"], prec_mz=row["prec_mz"], formula=row["formula"]
    ) if has_formula[row.name] else
    {"n_peaks": 0, "n_isotope_fa": np.nan, "frac_isotope_fa": np.nan, "has_isotope_fa": np.nan},
    axis=1,
).apply(pd.Series)

# Combine
comparison = pd.concat([
    fa_sample[["spec_id", "inst_type", "formula", "prec_mz"]],
    naive_stats[["n_peaks", "n_isotope", "has_isotope"]],
    fa_stats[["n_isotope_fa", "has_isotope_fa"]],
], axis=1)

# Summary (only rows with valid formula-aware results)
valid = comparison[has_formula].copy()
naive_pct = 100 * valid["has_isotope"].mean()
fa_pct = 100 * valid["has_isotope_fa"].mean()

print(f"\nNaive detection:           {naive_pct:.2f}% spectra with ≥1 isotope peak")
print(f"Formula-aware detection:   {fa_pct:.2f}% spectra with ≥1 isotope peak")
print(f"Reduction:                 {naive_pct - fa_pct:.2f} pp  "
      f"({100 * (naive_pct - fa_pct) / naive_pct:.1f}% fewer detections)")

# False-positive candidates: naive says yes, formula-aware says no
fp_candidates = valid[valid["has_isotope"] & ~valid["has_isotope_fa"]]
# True positives: both agree
tp = valid[valid["has_isotope"] & valid["has_isotope_fa"]]
print(f"\nBoth agree (isotope):       {len(tp):,}")
print(f"Naive only (likely FP):     {len(fp_candidates):,}")
print(f"Formula-aware only (rare):  {(valid['has_isotope_fa'] & ~valid['has_isotope']).sum():,}")

In [ ]:
# ── Plot: naive vs formula-aware by instrument type ───────────────────────────
grp_naive = valid.groupby("inst_type")["has_isotope"].mean().mul(100).rename("Naive")
grp_fa = valid.groupby("inst_type")["has_isotope_fa"].mean().mul(100).rename("Formula-aware")
grp = pd.concat([grp_naive, grp_fa], axis=1).sort_values("Naive", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(grp))
w = 0.35
ax.bar(x - w / 2, grp["Naive"], w, label="Naive", color="steelblue", alpha=0.85)
ax.bar(x + w / 2, grp["Formula-aware"], w, label="Formula-aware", color="darkorange", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(grp.index, rotation=20, ha="right")
ax.set_ylabel("% spectra with ≥1 isotope peak")
ax.set_title("NIST23: Naive vs Formula-aware isotope detection by instrument type")
ax.legend()
plt.tight_layout()
plt.show()

# ── Distribution of observed M+1/M+0 ratios vs expected ──────────────────────
# For spectra where naive says isotope AND we have formula+prec_mz
iso_rows = fa_sample[has_formula & naive_stats["has_isotope"].astype(bool)].head(2000)

obs_ratios, exp_ratios = [], []
for _, row in iso_rows.iterrows():
    mzs = np.array([p[0] for p in row["peaks"]], dtype=np.float64)
    ints = np.array([p[1] for p in row["peaks"]], dtype=np.float64)
    if len(mzs) == 0:
        continue
    mol_m1 = expected_m1_ratio(row["formula"])
    sort_idx = np.argsort(mzs)
    smzs, sints = mzs[sort_idx], ints[sort_idx]
    for i in range(len(smzs)):
        target = smzs[i] - NEUTRON_MASS
        lo = np.searchsorted(smzs, target - 0.02, side="left")
        hi = np.searchsorted(smzs, target + 0.02, side="right")
        if lo >= hi:
            continue
        m0 = sints[lo:hi].max()
        if m0 <= 0:
            continue
        obs = sints[i] / m0
        exp = max(mol_m1 * min(smzs[i] / row["prec_mz"], 1.0), 1e-4)
        obs_ratios.append(obs)
        exp_ratios.append(exp)

obs_ratios = np.array(obs_ratios)
exp_ratios = np.array(exp_ratios)
ratio_of_ratios = obs_ratios / exp_ratios  # 1.0 = perfect match

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].scatter(exp_ratios, obs_ratios, s=5, alpha=0.3, color="steelblue")
lim = max(exp_ratios.max(), obs_ratios.max()) * 1.05
axes[0].plot([0, lim], [0, lim], "r--", lw=1, label="obs = exp")
axes[0].plot([0, lim], [0, 3 * lim], "orange", lw=1, ls="--", label="3× expected (hi cutoff)")
axes[0].plot([0, lim], [0, 0.3 * lim], "green", lw=1, ls="--", label="0.3× expected (lo cutoff)")
axes[0].set_xlim(0, min(lim, 0.5))
axes[0].set_ylim(0, min(lim, 0.5))
axes[0].set_xlabel("Expected M+1/M+0 ratio")
axes[0].set_ylabel("Observed M+1/M+0 ratio")
axes[0].set_title("Expected vs Observed isotope intensity ratio")
axes[0].legend(fontsize=7)

axes[1].hist(np.log10(ratio_of_ratios.clip(1e-3, 1e3)), bins=60, color="slateblue", alpha=0.8, edgecolor="white")
axes[1].axvline(0, color="red", lw=1.5, ls="--", label="obs = exp")
axes[1].axvline(np.log10(3), color="orange", lw=1.5, ls="--", label="3× (hi cutoff)")
axes[1].axvline(np.log10(0.3), color="green", lw=1.5, ls="--", label="0.3× (lo cutoff)")
pct_in_window = 100 * np.mean((ratio_of_ratios >= 0.3) & (ratio_of_ratios <= 3.0))
axes[1].set_xlabel("log10(obs / exp) ratio")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Ratio-of-ratios distribution  [{pct_in_window:.1f}% within [0.3×, 3×] window]")
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

print(f"\nOf peaks naive-detected as isotopes: {pct_in_window:.1f}% pass the formula-aware ratio check")

## Cross-reference: NIST annotations vs m/z-spacing detector

`ann_df.ann_isotopes` contains NIST's own per-peak isotope labels (integer k: 0 = monoisotopic,
k > 0 = M+k) parsed from the annotation string (e.g. `C6H5+i`).  These are only available
for `nist20_hr` spectra with `[M+H]+` or `[M-H]-` precursors.

We join `ann_df` back to `spec_df` and ask:
- **Recall**: what fraction of NIST-labelled isotope peaks does the m/z detector find?
- **Precision**: what fraction of detector-flagged peaks are NIST-labelled isotopes?
- **Coverage gap**: how many spectra flagged by the detector have *no* annotation at all?

In [ ]:
print("Loading NIST20 ann_df…")
ann20 = pd.read_pickle(PROC_DP / "nist20/ann_df.pkl")[
    ["spec_id", "ann_peak_mzs", "ann_isotopes"]
].copy()
ann_spec_ids = set(ann20["spec_id"].unique())
print(f"  Annotated spectra: {len(ann_spec_ids):,}")

# Load only peaks for annotated spec_ids — avoids keeping the full spec_df in memory
print("  Loading peaks for annotated spectra only…")
_tmp = pd.read_pickle(PROC_DP / "nist20/spec_df.pkl")[["spec_id", "peaks"]]
ann_peaks_dict: dict = (
    _tmp[_tmp["spec_id"].isin(ann_spec_ids)]
    .set_index("spec_id")["peaks"]
    .to_dict()
)
del _tmp; gc.collect()
print(f"  Loaded peaks for {len(ann_peaks_dict):,} annotated spectra")

# Explode for per-peak counts only (keep small, free after)
ann20_peaks = ann20.explode(["ann_peak_mzs", "ann_isotopes"]).rename(
    columns={"ann_peak_mzs": "ann_mz", "ann_isotopes": "ann_iso_k"}
)
ann20_peaks["ann_iso_k"] = ann20_peaks["ann_iso_k"].astype(int)
ann20_peaks["is_ann_isotope"] = ann20_peaks["ann_iso_k"] != 0

n_ann_iso = ann20_peaks["is_ann_isotope"].sum()
n_ann_total = len(ann20_peaks)
print(f"  Total annotated peaks: {n_ann_total:,}")
print(f"  NIST-labelled isotope peaks: {n_ann_iso:,} ({100*n_ann_iso/n_ann_total:.2f}%)")
print(f"  Spectra with ≥1 annotated isotope peak: "
      f"{ann20_peaks[ann20_peaks['is_ann_isotope']]['spec_id'].nunique():,}")
del ann20_peaks; gc.collect()

In [ ]:
MZ_MATCH_TOL = 0.02

# ── Step 1: precompute detection masks for all annotated spectra ──────────────
# Sorted mzs required for searchsorted in step 2.
print(f"Precomputing detection masks for {len(ann_peaks_dict):,} spectra…")
det_dict: dict = {}   # spec_id -> (sorted_mzs, sorted_det_mask)
for spec_id, peaks in ann_peaks_dict.items():
    if not peaks:
        continue
    mzs = np.array([p[0] for p in peaks], dtype=np.float64)
    ints = np.array([p[1] for p in peaks], dtype=np.float64)
    mask = detect_isotope_peaks(mzs, ints)
    si = np.argsort(mzs)
    det_dict[spec_id] = (mzs[si], mask[si])
del ann_peaks_dict; gc.collect()
print(f"  Done. {len(det_dict):,} spectra with peaks.")

# ── Step 2: explode annotations → one row per annotated peak ─────────────────
ann20_exp = (
    ann20.explode(["ann_peak_mzs", "ann_isotopes"])
    .rename(columns={"ann_peak_mzs": "ann_mz", "ann_isotopes": "ann_iso_k"})
    .assign(ann_mz=lambda d: d["ann_mz"].astype(float),
            ann_iso_k=lambda d: d["ann_iso_k"].astype(int))
    [lambda d: d["spec_id"].isin(det_dict)]
    .reset_index(drop=True)
)
print(f"  Annotated peaks to match: {len(ann20_exp):,}")

# ── Step 3: vectorised nearest-neighbour match per spectrum ───────────────────
parts = []
for spec_id, grp in ann20_exp.groupby("spec_id", sort=False):
    smzs, smask = det_dict[spec_id]           # sorted mzs + detection mask
    amzs = grp["ann_mz"].to_numpy()

    # searchsorted gives insertion points; check both neighbours
    idx = np.searchsorted(smzs, amzs).clip(0, len(smzs) - 1)
    idx_lo = (idx - 1).clip(0, len(smzs) - 1)
    d_hi = np.abs(smzs[idx]    - amzs)
    d_lo = np.abs(smzs[idx_lo] - amzs)
    best = np.where(d_lo < d_hi, idx_lo, idx)
    best_d = np.minimum(d_hi, d_lo)

    valid = best_d <= MZ_MATCH_TOL
    if not valid.any():
        continue
    sub = grp[valid].copy()
    sub["det_isotope"] = smask[best[valid]]
    parts.append(sub[["spec_id", "ann_iso_k", "det_isotope"]])

xref_df = pd.concat(parts, ignore_index=True)
xref_df["is_ann_isotope"] = xref_df["ann_iso_k"] != 0
del ann20_exp, parts; gc.collect()

print(f"Matched {len(xref_df):,} annotated peaks")
print(f"  NIST isotope labels: {xref_df['is_ann_isotope'].sum():,}")
print(f"  Detector flags:      {xref_df['det_isotope'].sum():,}")

In [ ]:
# ── Confusion matrix & metrics ────────────────────────────────────────────────
tp = ((xref_df["is_ann_isotope"]) & (xref_df["det_isotope"])).sum()
fp = ((~xref_df["is_ann_isotope"]) & (xref_df["det_isotope"])).sum()
fn = ((xref_df["is_ann_isotope"]) & (~xref_df["det_isotope"])).sum()
tn = ((~xref_df["is_ann_isotope"]) & (~xref_df["det_isotope"])).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print("Confusion matrix (NIST annotation as ground truth):")
print(f"                   Det=isotope  Det=fragment")
print(f"  Ann=isotope  TP={tp:>8,}  FN={fn:>8,}   (NIST says isotope)")
print(f"  Ann=fragment FP={fp:>8,}  TN={tn:>8,}   (NIST says fragment)")
print()
print(f"Precision (of detector):  {precision:.3f}  — when detector fires, how often is NIST also isotope?")
print(f"Recall    (of detector):  {recall:.3f}  — what fraction of NIST isotopes does detector catch?")
print(f"F1:                       {f1:.3f}")
print()
print("Missed (FN) by isotope order k:")
print(xref_df[xref_df["is_ann_isotope"] & ~xref_df["det_isotope"]]["ann_iso_k"].value_counts().sort_index().to_string())

# ── Coverage gap: spectra with detector isotopes but no annotation ────────────
print()
det_iso_spec_ids = set(stats20[stats20["has_isotope"]]["spec_id"].dropna())
no_ann_but_detected = det_iso_spec_ids - ann_spec_ids
has_ann_and_detected = det_iso_spec_ids & ann_spec_ids
print(f"Spectra with detector isotopes:            {len(det_iso_spec_ids):,}")
print(f"  …that also have annotations:             {len(has_ann_and_detected):,} ({100*len(has_ann_and_detected)/max(1,len(det_iso_spec_ids)):.1f}%)")
print(f"  …with NO annotation at all (gap):        {len(no_ann_but_detected):,} ({100*len(no_ann_but_detected)/max(1,len(det_iso_spec_ids)):.1f}%)")

In [ ]:
# ── Plot: confusion breakdown + FN by isotope order ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: stacked bar showing TP/FP/FN/TN proportions
ax = axes[0]
total = tp + fp + fn + tn
categories = ["TP\n(both agree: isotope)", "FP\n(det only)", "FN\n(NIST only)", "TN\n(both agree: fragment)"]
values = [tp, fp, fn, tn]
colors = ["steelblue", "darkorange", "tomato", "mediumseagreen"]
bars = ax.bar(categories, values, color=colors, alpha=0.85)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.003,
            f"{val:,}\n({100*val/total:.1f}%)", ha="center", fontsize=8)
ax.set_ylabel("Number of peaks")
ax.set_title("Detector vs NIST annotation (peak level)")
ax.set_yscale("log")

# Right: FN breakdown by isotope order k
ax = axes[1]
fn_by_k = (
    xref_df[xref_df["is_ann_isotope"] & ~xref_df["det_isotope"]]["ann_iso_k"]
    .value_counts().sort_index()
)
tp_by_k = (
    xref_df[xref_df["is_ann_isotope"] & xref_df["det_isotope"]]["ann_iso_k"]
    .value_counts().sort_index()
)
all_ks = sorted(set(fn_by_k.index) | set(tp_by_k.index))
fn_vals = [fn_by_k.get(k, 0) for k in all_ks]
tp_vals = [tp_by_k.get(k, 0) for k in all_ks]
x = np.arange(len(all_ks))
w = 0.35
ax.bar(x - w/2, tp_vals, w, label="Detected (TP)", color="steelblue", alpha=0.85)
ax.bar(x + w/2, fn_vals, w, label="Missed (FN)", color="tomato", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f"M+{k}" for k in all_ks])
ax.set_ylabel("Number of peaks")
ax.set_title("Detected vs Missed by isotope order k")
ax.legend()

plt.suptitle("m/z-spacing detector vs NIST annotations (NIST20 HR, annotated spectra)", y=1.02)
plt.tight_layout()
plt.show()